### Install Dependencies 

In [ ]:
%pip install anthropic python-dotenv

### Load env variables

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### Create an API Client

In [ ]:
from antropic import Anthropic

client = Anthropic()
model="claude-sonnet-4-0"

### Make a Request 

In [ ]:
message = client.message.create(
    model=model,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "Write a poem about the beauty of nature within 5 lines."
        }
    ]
)

In [ ]:
message

In [ ]:
message.content[0].text

In [ ]:
## Multi-turn Conversation

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text


In [ ]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

### Chat Exercise - creating a chatbot through jupiter notebook

In [ ]:
# Make an initial list of messages
messages = []
# Use a 'while True' loop to run the chatbot forever
while True:
# Get user input
    user_input = input(">.")
    print(">", user_input)


# Add user input to the list of messages add
add_user_message(messages, user_input)
# Call Claude with the 'chat' function
answer = chat(messages)
# Add generated text to the list of messages
add_assistant_message(messages, answer)
# Print the generated text
print(answer)

### System Prompts :- Toning the style of response which claude generates

In [ ]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# Without system prompt
answer = chat(messages)

# With system prompt
system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""
answer = chat(messages, system=system)

### 
System Prompts Exercise:- 

- Write a python function that checks for system duplicates , since this generate lot of rough code you are supposed to pass system prompt as bring concise as possible, clean and optimized code 

- you are a python engineer you writes very concise code 

#### Temperature

In [ ]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

#### Testing temperature effects 

In [ ]:
# Low temperature - more predictable
answer = chat(messages, temperature=0.0)

# High temperature - more creative  
answer = chat(messages, temperature=1.0)

### Response Streaming :- 
Appear data coming in chunks as soon as its generated 

In [ ]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

#### Simplified text streaming 

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

In [ ]:
### Getting the final message 

In [ ]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        # Send each chunk to your client
        pass
    
    # Get the complete message for database storage
    final_message = stream.get_final_message()

#### Structured data 

In [ ]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])

In [ ]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())